# 04 — Exploratory Visualization

Matplotlib exploration charts with consistent display names, full race stratification, and closed database connections.

**Labeling:** All charts EXCEPT small multiples have value labels on bars. Small multiples are exploratory only (no labels).

**Display names:** Consistent short cause labels applied across all charts via `short_name()`.

**Chart inventory:**
- Chart 1: National abortion comparison (labeled bars)
- Chart 2: Top 7 causes by race (labeled horizontal bars, 1×6 grid)
- Chart 3: Top 10 causes by sex (stacked bars with percent labels, national)
- Chart 3b: Abortion comparison by race (White & Black, format matches Chart 1)
- Chart 4: Small multiples — top 10 causes × sex × age group (national) — NO labels
- Chart 5: Small multiples by race — top 7 causes × age per race (6 separate 2×4 grids) — NO labels
- Chart 6: Top causes by race & sex (stacked, percent labels, White & Black only)
- Analysis: Notable patterns from small multiples


## Setup & Config


In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingest import load_config
from src.clean_quality import get_connection, run_sql

cfg = load_config('config.yaml')
con = get_connection(cfg)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Display name mapping for consistent labeling across all charts
DISPLAY_NAMES = {
    'Diseases of heart': 'Heart disease',
    'Malignant neoplasms': 'Cancer',
    'Chronic lower respiratory diseases': 'Respiratory disease',
    'Cerebrovascular diseases': 'Stroke',
    'Alzheimer disease': "Alzheimer's",
    'Diabetes mellitus': 'Diabetes',
    'Accidents (unintentional injuries)': 'Accidents',
    'Intentional self-harm (suicide)': 'Suicide',
    'Chronic liver disease and cirrhosis': 'Liver disease',
    'Nephritis, nephrotic syndrome and nephrosis': 'Kidney disease',
    'Influenza and pneumonia': 'Flu/Pneumonia',
    'Essential hypertension and hypertensive renal disease': 'Hypertension',
    'Assault (homicide)': 'Homicide',
    'Pregnancy, childbirth and the puerperium': 'Pregnancy/childbirth',
}

def short_name(cause):
    """Map verbose cause name to short display name."""
    return DISPLAY_NAMES.get(cause, cause)

print('✓ Setup complete')


## Load Data


In [ ]:
# National aggregated data
mort_national = run_sql('SELECT * FROM mortality_national ORDER BY deaths DESC', con)
mort_by_sex_age = run_sql('SELECT * FROM mortality_by_sex_age', con)

# Race-stratified data
mort_race_sex = run_sql('SELECT * FROM mortality_race_sex', con)
mort_race_age = run_sql('SELECT * FROM mortality_race_age', con)

print(f'✓ National data: {len(mort_national)} causes')
print(f'✓ By sex+age (national): {len(mort_by_sex_age)} rows')
print(f'✓ By race+sex: {len(mort_race_sex)} rows')
print(f'✓ By race+age (no sex): {len(mort_race_age)} rows')

# Load export
try:
    master_table = pd.read_csv('export/abortion_cause_of_death_v1.csv')
    print(f'✓ Export loaded: {len(master_table)} rows')
except FileNotFoundError:
    print('⚠ Export file not found. Run 03-prepare.ipynb or scripts/generate_export.py first.')
    master_table = None


## Chart 1: National Abortion Comparison (Labeled)


In [ ]:
if master_table is None:
    print('⚠ Skipping Chart 1: export not available')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # LEFT: Without abortion (top 5)
    without = master_table[
        (master_table['scenario'] == 'Without abortion')
    ].head(5).sort_values('deaths', ascending=True).copy()
    without['cause_display'] = without['cause'].apply(short_name)
    
    ax1.barh(without['cause_display'], without['deaths'], color='steelblue')
    ax1.set_title('Leading Causes of Death, USA 2024 (All Persons)', fontsize=13, fontweight='bold', loc='left')
    ax1.set_xlabel('')
    ax1.set_xticks([])
    for i, (cause, v) in enumerate(zip(without['cause_display'], without['deaths'])):
        ax1.text(v + 10000, i, f'{v:,.0f}', va='center', fontsize=10, fontweight='bold')
    
    # RIGHT: With abortion - show the SAME top 5 causes + abortion
    with_abort = master_table[
        (master_table['scenario'] == 'With abortion')
    ].copy()
    top_5_codes = without['cause_code'].tolist()
    same_5_in_with = with_abort[with_abort['cause_code'].isin(top_5_codes)]
    abort_only = with_abort[with_abort['cause_code'] == 'ABORT']
    with_data = pd.concat([same_5_in_with, abort_only]).sort_values('deaths', ascending=True).copy()
    with_data['cause_display'] = with_data['cause'].apply(short_name)
    
    colors = ['red' if c == 'ABORT' else 'steelblue' for c in with_data['cause_code']]
    ax2.barh(with_data['cause_display'], with_data['deaths'], color=colors)
    ax2.set_title('If Abortion Were a Cause of Death', fontsize=13, fontweight='bold', loc='left')
    ax2.set_xlabel('')
    ax2.set_xticks([])
    for i, (cause, v) in enumerate(zip(with_data['cause_display'], with_data['deaths'])):
        ax2.text(v + 10000, i, f'{v:,.0f}', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('outputs/01_national_without_vs_with.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Chart 1: National comparison saved')


## Chart 2: Top Causes by Race (Labeled Horizontal Bars)


In [ ]:
# Clean race×sex data, aggregate by race (sum across sexes)
mort_race_clean = mort_race_sex[
    (mort_race_sex['icd_10_113_cause_list'].str.startswith('#', na=False)) &
    (~mort_race_sex['single_race_6'].isin(['Not Available', '']))
].copy()

# Extract cause name from the cause_code + cause_name field
race_cause = mort_race_clean.groupby(['single_race_6', 'icd_10_113_cause_list']).agg(
    {'deaths': 'sum'}
).reset_index().sort_values(['single_race_6', 'deaths'], ascending=[True, False])

race_cause['cause'] = race_cause['icd_10_113_cause_list'].str.replace('^#\s*', '', regex=True)
race_cause['cause'] = race_cause['cause'].str.replace(r'\s*\([^)]+\)\s*$', '', regex=True)
race_cause['cause_display'] = race_cause['cause'].apply(short_name)

races = sorted(mort_race_clean['single_race_6'].unique())
print(f'Races: {races}')

fig, axes = plt.subplots(1, 6, figsize=(20, 6))
for idx, race in enumerate(races):
    ax = axes[idx]
    race_data = race_cause[
        race_cause['single_race_6'] == race
    ].head(7).sort_values('deaths', ascending=True)
    
    ax.barh(race_data['cause_display'], race_data['deaths'], color='steelblue')
    ax.set_title(race, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticks([])
    # Add value labels
    for i, (cause, v) in enumerate(zip(race_data['cause_display'], race_data['deaths'])):
        ax.text(v + 1000, i, f'{v:,.0f}', va='center', fontsize=8, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Top 7 Causes of Death by Race (2024)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/02_causes_by_race.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart 2: Top causes by race saved')


## Chart 3: Top Causes by Sex (Stacked Bars with Percent Labels, National)


In [ ]:
# mort_by_sex_age already has clean cause names (no # prefix)
top_by_sex = mort_by_sex_age.groupby(['sex', 'cause']).agg(
    {'deaths': 'sum'}
).reset_index().sort_values(['sex', 'deaths'], ascending=[True, False])

# Get top 10 causes (aggregate by cause, sum across sexes)
top_10_causes_nat = top_by_sex.groupby('cause')['deaths'].sum().nlargest(10).index.tolist()

# Pivot to sex × cause
sex_cause_pivot = top_by_sex[top_by_sex['cause'].isin(top_10_causes_nat)].pivot_table(
    index='cause',
    columns='sex',
    values='deaths',
    aggfunc='sum',
    fill_value=0
)

# Sort by total (Female + Male) descending
sex_cause_pivot['total'] = sex_cause_pivot.sum(axis=1)
sex_cause_pivot = sex_cause_pivot.sort_values('total', ascending=True)
sex_cause_pivot = sex_cause_pivot.drop('total', axis=1)

# Ensure order: Female, Male
if 'Female' in sex_cause_pivot.columns and 'Male' in sex_cause_pivot.columns:
    sex_cause_pivot = sex_cause_pivot[['Female', 'Male']]

# Apply display names to index
sex_cause_pivot.index = sex_cause_pivot.index.map(short_name)

# Create stacked bar chart
fig, ax = plt.subplots(figsize=(14, 7))
sex_cause_pivot.plot(
    kind='barh',
    stacked=True,
    ax=ax,
    color=['indianred', 'steelblue'],
    legend=True
)

ax.set_title('Top 10 Causes of Death by Sex (National)', fontsize=13, fontweight='bold', loc='left')
ax.set_xlabel('')
ax.set_xticks([])
ax.legend(title='Sex', loc='lower right', fontsize=10, title_fontsize=10)

# Add total labels and percent labels on right side
for i, cause in enumerate(sex_cause_pivot.index):
    female_val = sex_cause_pivot.loc[cause, 'Female']
    male_val = sex_cause_pivot.loc[cause, 'Male']
    total_val = female_val + male_val
    female_pct = 100 * female_val / total_val if total_val > 0 else 0
    male_pct = 100 * male_val / total_val if total_val > 0 else 0
    
    # Total on the right
    ax.text(total_val + 10000, i, f'{total_val:,.0f}', va='center', fontsize=10, fontweight='bold')
    # Percent labels within bar
    ax.text(female_val / 2, i, f'{female_pct:.0f}%', va='center', ha='center', fontsize=8, fontweight='bold', color='white')
    ax.text(female_val + male_val / 2, i, f'{male_pct:.0f}%', va='center', ha='center', fontsize=8, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig('outputs/03_causes_by_sex_national.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart 3: Top causes by sex (national, stacked) saved')


## Chart 3b: National Abortion Comparison by Race (White & Black)

Replicate Chart 1 for White and Black races (2 figures, matching the national comparison format).


In [ ]:
# Create race-specific comparisons for White and Black
if master_table is None:
    print('⚠ Skipping Chart 3b: export not available')
else:
    races_to_compare = ['White', 'Black or African American']
    
    for race in races_to_compare:
        # Filter for this race, aggregate across sexes
        race_sex_data = mort_race_sex[
            (mort_race_sex['icd_10_113_cause_list'].str.startswith('#', na=False)) &
            (mort_race_sex['single_race_6'] == race) &
            (~mort_race_sex['sex'].isin(['Not Available', '']))
        ].copy()
        
        race_sex_data['cause'] = race_sex_data['icd_10_113_cause_list'].str.replace('^#\s*', '', regex=True)
        race_sex_data['cause'] = race_sex_data['cause'].str.replace(r'\s*\([^)]+\)\s*$', '', regex=True)
        
        race_totals = race_sex_data.groupby('cause')['deaths'].sum().reset_index().sort_values('deaths', ascending=False)
        race_totals['cause_display'] = race_totals['cause'].apply(short_name)
        
        # Get top 5 for this race
        top_5_race = race_totals.head(5).copy()
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # LEFT: Top 5 without abortion
        top_5_sorted = top_5_race.sort_values('deaths', ascending=True)
        ax1.barh(top_5_sorted['cause_display'], top_5_sorted['deaths'], color='steelblue')
        ax1.set_title(f'Leading Causes of Death, {race}, 2024', fontsize=13, fontweight='bold', loc='left')
        ax1.set_xlabel('')
        ax1.set_xticks([])
        for i, (cause, v) in enumerate(zip(top_5_sorted['cause_display'], top_5_sorted['deaths'])):
            ax1.text(v + max(top_5_sorted['deaths']) * 0.02, i, f'{v:,.0f}', va='center', fontsize=10, fontweight='bold')
        
        # RIGHT: Top 5 + hypothetical abortion
        # Estimate abortion for this race using proportional deaths
        race_death_prop = race_sex_data['deaths'].sum() / mort_by_sex_age['deaths'].sum()
        national_abort = master_table[master_table['cause_code'] == 'ABORT']['deaths'].values[0]
        race_abort_est = national_abort * race_death_prop
        
        with_abort_data = pd.concat([
            top_5_race[['cause', 'deaths', 'cause_display']].reset_index(drop=True),
            pd.DataFrame({'cause': ['Abortion'], 'deaths': [race_abort_est], 'cause_display': ['Abortion']})
        ]).sort_values('deaths', ascending=True)
        
        colors = ['red' if c == 'Abortion' else 'steelblue' for c in with_abort_data['cause_display']]
        ax2.barh(with_abort_data['cause_display'], with_abort_data['deaths'], color=colors)
        ax2.set_title(f'If Abortion Were a Cause of Death, {race}, 2024', fontsize=13, fontweight='bold', loc='left')
        ax2.set_xlabel('')
        ax2.set_xticks([])
        for i, (cause, v) in enumerate(zip(with_abort_data['cause_display'], with_abort_data['deaths'])):
            ax2.text(v + max(with_abort_data['deaths']) * 0.02, i, f'{v:,.0f}', va='center', fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        safe_race = race.replace('/', '_').replace(' ', '_').lower()
        plt.savefig(f'outputs/03b_national_without_vs_with_race_{safe_race}.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    print('✓ Chart 3b: Abortion comparisons by race (White, Black) saved')


## Chart 4: Small Multiples by Cause × Sex × Age (National) — NO LABELS

Horizontal stacked bars showing sex distribution by age group for each of the top 10 causes.


In [ ]:
# Get top 10 causes at national level
top_10_causes = mort_national.head(10)['cause'].tolist()

# mort_by_sex_age already has clean cause names
top_10_data = mort_by_sex_age[mort_by_sex_age['cause'].isin(top_10_causes)].copy()

age_order = [
    'Under 1 year', '1-4 years', '5-9 years', '10-14 years', '15-19 years',
    '20-24 years', '25-29 years', '30-34 years', '35-39 years', '40-44 years',
    '45-49 years', '50-54 years', '55-59 years', '60-64 years', '65-69 years',
    '70-74 years', '75-79 years', '80-84 years', '85 years and over'
]

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for idx, cause in enumerate(top_10_causes):
    ax = axes[idx]
    cause_data = top_10_data[top_10_data['cause'] == cause].copy()
    
    # Pivot: age × sex
    pivot = cause_data.pivot_table(
        index='age_group',
        columns='sex',
        values='deaths',
        aggfunc='sum',
        fill_value=0
    )
    
    # Reorder by age
    pivot = pivot.reindex([ag for ag in age_order if ag in pivot.index])
    
    # Ensure correct column order (Male, Female)
    if 'Male' in pivot.columns and 'Female' in pivot.columns:
        pivot = pivot[['Male', 'Female']]
    
    # Stacked horizontal bar chart
    pivot.plot(
        kind='barh',
        stacked=True,
        ax=ax,
        color=['steelblue', 'indianred'],
        legend=(idx == 0)
    )
    
    display_cause = short_name(cause)
    ax.set_title(display_cause, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='both', labelsize=7)
    
    if idx == 0:
        ax.legend(title='Sex', fontsize=8, title_fontsize=8, loc='lower right')
    else:
        ax.legend().remove()

plt.suptitle('Top 10 Causes of Death: Sex Distribution by Age Group — National (2024)',
             fontsize=12, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('outputs/04_causes_sex_age_multiples_national.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart 4: Small multiples (cause × sex × age, national) saved')


## Chart 5: Small Multiples by Race (Top Causes × Age) — NO LABELS

Note: `mortality_race_age` lacks sex column (aggregated), so we show age distribution only.
These are exploratory, showing top causes per race by age group.


In [ ]:
# Clean race×age data
race_age_clean = mort_race_age[
    (mort_race_age['icd_10_113_cause_list'].str.startswith('#', na=False)) &
    (~mort_race_age['single_race_6'].isin(['Not Available', '']))
].copy()

race_age_clean['cause'] = race_age_clean['icd_10_113_cause_list'].str.replace('^#\s*', '', regex=True)
race_age_clean['cause'] = race_age_clean['cause'].str.replace(r'\s*\([^)]+\)\s*$', '', regex=True)
race_age_clean['cause_display'] = race_age_clean['cause'].apply(short_name)

races_list = sorted(race_age_clean['single_race_6'].unique())
print(f'Creating small multiples for {len(races_list)} races')

for race_idx, race in enumerate(races_list):
    race_data = race_age_clean[race_age_clean['single_race_6'] == race].copy()
    
    # Get top 7 causes by total deaths in this race
    top_causes_by_race = race_data.groupby('cause')['deaths'].sum().nlargest(7).index.tolist()
    
    # Filter to top causes
    top_race_data = race_data[race_data['cause'].isin(top_causes_by_race)].copy()
    
    # Create 2×4 grid for this race (7 causes + 1 empty)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()
    
    for ax_idx, cause in enumerate(top_causes_by_race):
        ax = axes[ax_idx]
        cause_data = top_race_data[top_race_data['cause'] == cause].copy()
        
        # Pivot: age (just deaths, no sex column in race_age)
        pivot = cause_data.pivot_table(
            index='five_year_age_groups',
            values='deaths',
            aggfunc='sum'
        )
        
        # Reindex by age order (only include ages that exist in pivot)
        available_ages = [ag for ag in age_order if ag in pivot.index]
        if available_ages:
            pivot = pivot.reindex(available_ages)
        
        # Horizontal bar (no stacking, no labels)
        ax.barh(pivot.index, pivot.values.flatten(), color='steelblue')
        display_cause = short_name(cause)
        ax.set_title(display_cause, fontsize=9, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.tick_params(axis='both', labelsize=7)
        ax.set_xticks([])
    
    # Hide the last empty subplot
    axes[7].axis('off')
    
    plt.suptitle(f'Top Causes of Death by Age Group — {race} (2024)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    safe_race = race.replace('/', '_').replace(' ', '_').lower()
    plt.savefig(f'outputs/05_causes_age_multiples_race_{safe_race}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('✓ Chart 5: Small multiples by race saved (6 figures)')


## Chart 6: Top Causes by Race & Sex (Stacked, Percent Labels) — White & Black Only

Replicate the stacked format from Chart 3 for White and Black races.


In [ ]:
# Clean race×sex data
race_sex_clean = mort_race_sex[
    (mort_race_sex['icd_10_113_cause_list'].str.startswith('#', na=False)) &
    (~mort_race_sex['single_race_6'].isin(['Not Available', ''])) &
    (~mort_race_sex['sex'].isin(['Not Available', '']))
].copy()

race_sex_clean['cause'] = race_sex_clean['icd_10_113_cause_list'].str.replace('^#\s*', '', regex=True)
race_sex_clean['cause'] = race_sex_clean['cause'].str.replace(r'\s*\([^)]+\)\s*$', '', regex=True)

races_to_chart = ['White', 'Black or African American']

for race in races_to_chart:
    race_data = race_sex_clean[race_sex_clean['single_race_6'] == race].copy()
    
    # Get top 10 causes for this race
    top_10_race = race_data.groupby('cause')['deaths'].sum().nlargest(10).index.tolist()
    
    # Pivot to sex × cause
    sex_cause_pivot = race_data[race_data['cause'].isin(top_10_race)].pivot_table(
        index='cause',
        columns='sex',
        values='deaths',
        aggfunc='sum',
        fill_value=0
    )
    
    # Sort by total descending
    sex_cause_pivot['total'] = sex_cause_pivot.sum(axis=1)
    sex_cause_pivot = sex_cause_pivot.sort_values('total', ascending=True)
    sex_cause_pivot = sex_cause_pivot.drop('total', axis=1)
    
    # Ensure order: Female, Male
    if 'Female' in sex_cause_pivot.columns and 'Male' in sex_cause_pivot.columns:
        sex_cause_pivot = sex_cause_pivot[['Female', 'Male']]
    
    # Apply display names
    sex_cause_pivot.index = sex_cause_pivot.index.map(short_name)
    
    # Create stacked bar chart
    fig, ax = plt.subplots(figsize=(14, 8))
    sex_cause_pivot.plot(
        kind='barh',
        stacked=True,
        ax=ax,
        color=['indianred', 'steelblue'],
        legend=True
    )
    
    ax.set_title(f'Top 10 Causes of Death by Sex — {race}', fontsize=13, fontweight='bold', loc='left')
    ax.set_xlabel('')
    ax.set_xticks([])
    ax.legend(title='Sex', loc='lower right', fontsize=10, title_fontsize=10)
    
    # Add total and percent labels
    for i, cause in enumerate(sex_cause_pivot.index):
        female_val = sex_cause_pivot.loc[cause, 'Female']
        male_val = sex_cause_pivot.loc[cause, 'Male']
        total_val = female_val + male_val
        female_pct = 100 * female_val / total_val if total_val > 0 else 0
        male_pct = 100 * male_val / total_val if total_val > 0 else 0
        
        ax.text(total_val + max(sex_cause_pivot.sum(axis=1)) * 0.01, i, f'{total_val:,.0f}', 
                va='center', fontsize=10, fontweight='bold')
        ax.text(female_val / 2, i, f'{female_pct:.0f}%', va='center', ha='center', 
                fontsize=8, fontweight='bold', color='white')
        ax.text(female_val + male_val / 2, i, f'{male_pct:.0f}%', va='center', ha='center', 
                fontsize=8, fontweight='bold', color='white')
    
    plt.tight_layout()
    safe_race = race.replace('/', '_').replace(' ', '_').lower()
    plt.savefig(f'outputs/06_causes_by_sex_race_{safe_race}_stacked.png', dpi=150, bbox_inches='tight')
    plt.show()

print('✓ Chart 6: Top causes by sex (White & Black, stacked) saved')


## Analysis: Notable Patterns from Small Multiples

Review interesting data points from Charts 4 and 5 that might be worth individual social media charts.


In [ ]:
print("\n" + "="*70)
print("NOTABLE PATTERNS FROM SMALL MULTIPLES")
print("="*70)

# CHART 4 ANALYSIS: Sex × Age by Cause (National)
print("\n### CHART 4: Sex × Age by Cause (National) ###\n")

# Find causes where one sex dominates
for cause in top_10_causes:
    cause_data = top_10_data[top_10_data['cause'] == cause].copy()
    by_sex = cause_data.groupby('sex')['deaths'].sum()
    
    if 'Female' in by_sex.index and 'Male' in by_sex.index:
        female_pct = 100 * by_sex['Female'] / by_sex.sum()
        male_pct = 100 * by_sex['Male'] / by_sex.sum()
        
        display = short_name(cause)
        
        # Flag interesting patterns
        if female_pct > 55:
            print(f"✓ {display}: {female_pct:.1f}% Female — predominantly female")
        elif male_pct > 55:
            print(f"✓ {display}: {male_pct:.1f}% Male — predominantly male")

# CHART 5 ANALYSIS: Top Causes by Race
print("\n### CHART 5: Top Causes by Race ###\n")

# Check for causes that rank differently by race
race_rankings = {}
for race in races_list:
    race_data = mort_race_sex[
        (mort_race_sex['icd_10_113_cause_list'].str.startswith('#', na=False)) &
        (mort_race_sex['single_race_6'] == race) &
        (~mort_race_sex['sex'].isin(['Not Available', '']))
    ].copy()
    
    race_data['cause'] = race_data['icd_10_113_cause_list'].str.replace('^#\s*', '', regex=True)
    race_data['cause'] = race_data['cause'].str.replace(r'\s*\([^)]+\)\s*$', '', regex=True)
    
    top_causes = race_data.groupby('cause')['deaths'].sum().nlargest(7).index.tolist()
    race_rankings[race] = top_causes

# Find causes that appear in top 7 for some races but not others
all_top_causes = set()
for causes in race_rankings.values():
    all_top_causes.update(causes)

for cause in all_top_causes:
    races_with_cause = [r for r, causes in race_rankings.items() if cause in causes]
    if len(races_with_cause) > 0 and len(races_with_cause) < len(races_list):
        display = short_name(cause)
        print(f"✓ {display}: In top 7 for {', '.join(races_with_cause)}")

print("\n" + "="*70)
print("Consider pulling individual charts for notable patterns above.")
print("="*70)


## Template Test: Stacked Horizontal Bar (Top 10 by Sex)

Validates the reusable `stacked_horizontal_bar()` template from `shared/chart_templates.py`.
This is the standard chart pattern for all future stacked bar charts.

In [ ]:
sys.path.insert(0, str(PROJECT.parent / "shared"))
from chart_templates import stacked_horizontal_bar
from src.viz_social import save_social

# Build df: top 10 causes with male/female death counts
top10 = con.execute("""
    SELECT cause_raw, cause as cause_clean, deaths
    FROM mortality_national ORDER BY deaths DESC LIMIT 10
""").df()
top10["cause"] = top10["cause_clean"].map(lambda c: DISPLAY_NAMES.get(c, c))

cause_raw_list = top10["cause_raw"].tolist()
placeholders = ",".join([f"'{c}'" for c in cause_raw_list])
sex_by_cause = con.execute(f"""
    SELECT icd_10_113_cause_list as cause_raw, sex, SUM(deaths) as deaths
    FROM mortality_sex_age
    WHERE icd_10_113_cause_list IN ({placeholders})
    GROUP BY icd_10_113_cause_list, sex
""").df()

rows = []
for _, row in top10.iterrows():
    sex_data = sex_by_cause[sex_by_cause["cause_raw"] == row["cause_raw"]]
    male_d = int(sex_data[sex_data["sex"] == "Male"]["deaths"].sum())
    female_d = int(sex_data[sex_data["sex"] == "Female"]["deaths"].sum())
    rows.append({"cause": row["cause"], "male_deaths": male_d, "female_deaths": female_d})

df_stacked = pd.DataFrame(rows)
print(df_stacked)

# Generate chart using template
chart = stacked_horizontal_bar(
    df_stacked,
    category_col="cause",
    segments=[
        {"value_col": "male_deaths", "label": "Male", "color": "#005F73"},
        {"value_col": "female_deaths", "label": "Female", "color": "#E9D8A6"},
    ],
    title="Top 10 Leading Causes of Death by Sex (2024)",
    source="CDC WONDER Underlying Cause of Death, 2024",
)

import yaml
with open(PROJECT / "config.yaml") as f:
    viz_cfg = yaml.safe_load(f)

save_social(chart, viz_cfg, "template_top10_by_sex", preset="twitter_landscape")
print("Template chart exported to outputs/social/template_top10_by_sex.png")

from IPython.display import Image, display
display(Image(filename=str(PROJECT / "outputs" / "social" / "template_top10_by_sex.png")))

## Supplemental Data Exploration: Cause Subtypes & Homicide Demographics

Using the new `cause_detail_by_sex_race` (WONDER subtypes) and
`shr_homicides_2024` (FBI case-level) tables.

In [ ]:
# === Accident Subtypes by Sex ===
# What makes up the "Accidents" bar? Poisoning (overdoses) vs motor vehicle vs falls

accident_subtypes = con.execute("""
  SELECT 
    icd_10_113_cause_list as cause,
    sex,
    SUM(deaths) as deaths,
    category
  FROM cause_detail_by_sex_race
  WHERE category = 'accidents_subtype'
    AND single_race_6 NOT IN ('Not Available', 'More than one race')
  GROUP BY icd_10_113_cause_list, sex, category
  ORDER BY deaths DESC
""").df()

# Clean cause names for display
ACCIDENT_NAMES = {
    'Accidental poisoning and exposure to noxious substances (X40-X49)': 'Poisoning (overdoses)',
    'Motor vehicle accidents': 'Motor vehicle',
    'Falls (W00-W19)': 'Falls',
    'Accidental drowning and submersion (W65-W74)': 'Drowning',
    'Accidental exposure to smoke, fire and flames (X00-X09)': 'Fire/smoke',
    'Accidental discharge of firearms (W32-W34)': 'Firearms (accidental)',
}

def short_accident(cause):
    for key, val in ACCIDENT_NAMES.items():
        if key in cause:
            return val
    if 'Motor vehicle' in cause:
        return 'Motor vehicle'
    if 'Transport' in cause:
        return 'Transport (other)'
    if 'Nontransport' in cause:
        return 'Nontransport (total)'
    if 'Other and unspecified' in cause:
        return 'Other/unspecified'
    if 'Water, air' in cause:
        return 'Water/air transport'
    if 'Other land' in cause:
        return 'Other land transport'
    return cause[:40]

accident_subtypes['cause_short'] = accident_subtypes['cause'].apply(short_accident)

# National totals by subtype (sum across races)
acc_national = accident_subtypes.groupby(['cause_short', 'sex'])['deaths'].sum().reset_index()
acc_national_total = acc_national.groupby('cause_short')['deaths'].sum().sort_values(ascending=False)

print('=== ACCIDENT SUBTYPES (National, 2024) ===')
print(f'{"Cause":<25} {"Total":>10} {"Male":>10} {"Female":>10} {"% Male":>8}')
print('-' * 70)
for cause in acc_national_total.index:
    total = int(acc_national_total[cause])
    male = int(acc_national[(acc_national['cause_short']==cause) & (acc_national['sex']=='Male')]['deaths'].sum())
    female = int(acc_national[(acc_national['cause_short']==cause) & (acc_national['sex']=='Female')]['deaths'].sum())
    pct_male = round(100*male/total) if total > 0 else 0
    print(f'{cause:<25} {total:>10,} {male:>10,} {female:>10,} {pct_male:>7}%')

# Key stat for annotation
total_accidents = int(acc_national_total.sum())
poisoning = int(acc_national_total.get('Poisoning (overdoses)', 0))
motor = int(acc_national_total.get('Motor vehicle', 0))
print(f'\n  Key stats:')
print(f'    Poisoning (overdoses): {poisoning:,} ({round(100*poisoning/total_accidents)}% of accidents)')
print(f'    Motor vehicle: {motor:,} ({round(100*motor/total_accidents)}% of accidents)')

In [ ]:
# === Suicide & Homicide: Firearms % by Sex and Race ===

firearms_data = con.execute("""
  SELECT 
    category,
    icd_10_113_cause_list as cause,
    sex,
    single_race_6 as race,
    deaths
  FROM cause_detail_by_sex_race
  WHERE (category LIKE 'suicide%' OR category LIKE 'homicide%')
    AND single_race_6 NOT IN ('Not Available', 'More than one race')
  ORDER BY category, sex, single_race_6
""").df()

# Separate totals from subtypes
totals = firearms_data[firearms_data['category'].str.endswith('_total')].copy()
subtypes = firearms_data[firearms_data['category'].str.endswith('_subtype')].copy()

# Mark firearms vs other
subtypes['is_firearms'] = subtypes['cause'].str.contains('firearms')

# Compute firearms % for suicide by sex
print('=== SUICIDE: Firearms % ===')
suicide_totals = totals[totals['category']=='suicide_total'].groupby('sex')['deaths'].sum()
suicide_firearms = subtypes[(subtypes['category']=='suicide_subtype') & (subtypes['is_firearms'])].groupby('sex')['deaths'].sum()
for sex in ['Male', 'Female']:
    total = int(suicide_totals.get(sex, 0))
    firearms = int(suicide_firearms.get(sex, 0))
    pct = round(100*firearms/total) if total > 0 else 0
    print(f'  {sex}: {firearms:,} / {total:,} = {pct}% firearms')

# By race
print('\n  By race (both sexes):')
suicide_totals_race = totals[totals['category']=='suicide_total'].groupby('race')['deaths'].sum()
suicide_firearms_race = subtypes[(subtypes['category']=='suicide_subtype') & (subtypes['is_firearms'])].groupby('race')['deaths'].sum()
for race in ['White', 'Black or African American', 'Asian', 'American Indian or Alaska Native']:
    total = int(suicide_totals_race.get(race, 0))
    firearms = int(suicide_firearms_race.get(race, 0))
    pct = round(100*firearms/total) if total > 0 else 0
    print(f'    {race}: {firearms:,} / {total:,} = {pct}% firearms')

# Homicide firearms %
print('\n=== HOMICIDE: Firearms % ===')
hom_totals = totals[totals['category']=='homicide_total'].groupby('sex')['deaths'].sum()
hom_firearms = subtypes[(subtypes['category']=='homicide_subtype') & (subtypes['is_firearms'])].groupby('sex')['deaths'].sum()
for sex in ['Male', 'Female']:
    total = int(hom_totals.get(sex, 0))
    firearms = int(hom_firearms.get(sex, 0))
    pct = round(100*firearms/total) if total > 0 else 0
    print(f'  {sex}: {firearms:,} / {total:,} = {pct}% firearms')

print('\n  By race (both sexes):')
hom_totals_race = totals[totals['category']=='homicide_total'].groupby('race')['deaths'].sum()
hom_firearms_race = subtypes[(subtypes['category']=='homicide_subtype') & (subtypes['is_firearms'])].groupby('race')['deaths'].sum()
for race in ['White', 'Black or African American', 'Asian', 'American Indian or Alaska Native']:
    total = int(hom_totals_race.get(race, 0))
    firearms = int(hom_firearms_race.get(race, 0))
    pct = round(100*firearms/total) if total > 0 else 0
    print(f'    {race}: {firearms:,} / {total:,} = {pct}% firearms')

In [ ]:
# === FBI SHR: Victim × Offender Race Cross-Tab (2024) ===

# Exclude unknown offenders for cleaner cross-tab
cross_tab = con.execute("""
  SELECT 
    VicRace as victim_race,
    OffRace as offender_race,
    COUNT(*) as cases
  FROM shr_homicides_2024
  WHERE OffRace != 'Unknown'
    AND VicRace IN ('White', 'Black')
    AND OffRace IN ('White', 'Black')
  GROUP BY VicRace, OffRace
  ORDER BY VicRace, cases DESC
""").df()

print('=== VICTIM × OFFENDER RACE (2024, White & Black only) ===')
print(cross_tab.to_string(index=False))

# Compute intra-racial %
print('\n  Intra-racial homicide rates:')
for vic_race in ['White', 'Black']:
    subset = cross_tab[cross_tab['victim_race'] == vic_race]
    total = subset['cases'].sum()
    same_race = int(subset[subset['offender_race'] == vic_race]['cases'].iloc[0])
    pct = round(100 * same_race / total) if total > 0 else 0
    print(f'    {vic_race} victims killed by {vic_race} offenders: {same_race:,} / {total:,} = {pct}%')

# Weapon breakdown
print('\n=== WEAPON BREAKDOWN (2024) ===')
weapons = con.execute("""
  SELECT 
    CASE 
      WHEN Weapon LIKE '%pistol%' OR Weapon LIKE '%Handgun%' THEN 'Handgun'
      WHEN Weapon LIKE '%Rifle%' THEN 'Rifle'
      WHEN Weapon LIKE '%Shotgun%' THEN 'Shotgun'
      WHEN Weapon LIKE '%Firearm%' OR Weapon LIKE '%gun%' THEN 'Firearm (type unknown)'
      WHEN Weapon LIKE '%Knife%' OR Weapon LIKE '%cutting%' THEN 'Knife/cutting'
      WHEN Weapon LIKE '%Personal%' OR Weapon LIKE '%beating%' THEN 'Hands/feet/beating'
      WHEN Weapon LIKE '%Blunt%' THEN 'Blunt object'
      ELSE 'Other/unknown'
    END as weapon_group,
    COUNT(*) as cases
  FROM shr_homicides_2024
  GROUP BY weapon_group
  ORDER BY cases DESC
""").df()
total_cases = weapons['cases'].sum()
weapons['pct'] = (100 * weapons['cases'] / total_cases).round(1)
print(weapons.to_string(index=False))

firearms_total = weapons[weapons['weapon_group'].str.contains('Handgun|Rifle|Shotgun|Firearm')]['cases'].sum()
print(f'\n  All firearms combined: {firearms_total:,} / {total_cases:,} = {round(100*firearms_total/total_cases)}%')

In [ ]:
# === Relationship & Circumstance (2024) ===

print('=== VICTIM-OFFENDER RELATIONSHIP ===')
relationships = con.execute("""
  SELECT 
    Relationship,
    COUNT(*) as cases
  FROM shr_homicides_2024
  WHERE Relationship != 'Relationship not determined'
  GROUP BY Relationship
  ORDER BY cases DESC
  LIMIT 12
""").df()
total_known = relationships['cases'].sum()
relationships['pct'] = (100 * relationships['cases'] / total_known).round(1)
print(relationships.to_string(index=False))

# Intimate partner violence stats
ipv_terms = ['Wife', 'Husband', 'Girlfriend', 'Boyfriend', 'Common-law wife', 'Common-law husband', 'Ex-wife', 'Ex-husband']
ipv = con.execute(f"""
  SELECT 
    VicSex as victim_sex,
    COUNT(*) as cases
  FROM shr_homicides_2024
  WHERE Relationship IN ({','.join([f"'{t}'" for t in ipv_terms])})
  GROUP BY VicSex
""").df()
print(f'\n  Intimate partner homicides by victim sex:')
print(ipv.to_string(index=False))

print('\n=== TOP CIRCUMSTANCES ===')
circumstances = con.execute("""
  SELECT 
    Circumstance,
    COUNT(*) as cases
  FROM shr_homicides_2024
  WHERE Circumstance NOT IN ('All other manslaughter by negligence', 'Undetermined', '')
  GROUP BY Circumstance
  ORDER BY cases DESC
  LIMIT 10
""").df()
print(circumstances.to_string(index=False))

## Trend Chart: Top 5 Causes of Death + Abortion Over Time (Rate per 100k, 1999-2024)

Line chart showing death **rates per 100,000 population** for the top 5 causes each year
(dynamic ranking), with abortion overlaid for scale.
Using rates controls for population growth (~22% increase over the period).
Lines stop when a cause drops out of the top 5 and resume if it re-enters.

In [ ]:
# === Trend Line Chart: Top 5 Causes (Rate per 100k) + Abortion (1999-2024) ===
# Uses crude death rate per 100,000 population to control for population growth.
# Top 5 determined per year by rate. Lines stop when a cause exits the top 5.
import matplotlib.pyplot as plt
import numpy as np

# Get all causes by year with rate (filter out rows with no population)
all_trend = con.execute("""
  SELECT year, icd_10_113_cause_list as cause, deaths, population, crude_rate
  FROM mortality_trend_national
  WHERE icd_10_113_cause_list LIKE '#%'
    AND population > 0
    AND crude_rate IS NOT NULL
  ORDER BY year, crude_rate DESC
""").df()

# Rank within each year by rate and keep top 5
all_trend['rank'] = all_trend.groupby('year')['crude_rate'].rank(method='first', ascending=False)
top5_dynamic = all_trend[all_trend['rank'] <= 5].copy()

# All causes that ever appear in top 5
ever_top5 = sorted(top5_dynamic['cause'].unique())
print(f'Causes that appear in any year\'s top 5 (by rate): {len(ever_top5)}')
for c in ever_top5:
    yrs = sorted(top5_dynamic[top5_dynamic['cause'] == c]['year'].unique())
    print(f'  {c[:55]}  ({len(yrs)} yrs)')

# Get abortion trend + compute rate using same population
abort_trend = con.execute("""
  SELECT a.year, a.abortions,
         m.population,
         (a.abortions * 1.0 / m.population) * 100000 as rate_per_100k
  FROM abortions_trend a
  JOIN (
    SELECT DISTINCT year, population
    FROM mortality_trend_national
    WHERE population > 0
  ) m ON a.year = m.year
  WHERE a.year >= 1999
""").df()

print(f'\nAbortion rate range: {abort_trend["rate_per_100k"].min():.1f} - {abort_trend["rate_per_100k"].max():.1f} per 100k')

# Display names
CAUSE_NAMES = {
    '#Diseases of heart (I00-I09,I11,I13,I20-I51)': 'Heart disease',
    '#Malignant neoplasms (C00-C97)': 'Cancer',
    '#Cerebrovascular diseases (I60-I69)': 'Stroke',
    '#Accidents (unintentional injuries) (V01-X59,Y85-Y86)': 'Accidents',
    '#Chronic lower respiratory diseases (J40-J47)': 'Respiratory disease',
    '#COVID-19 (U07.1)': 'COVID-19',
    '#Alzheimer disease (G30)': "Alzheimer's",
    '#Diabetes mellitus (E10-E14)': 'Diabetes',
    '#Influenza and pneumonia (J09-J18)': 'Flu/Pneumonia',
    '#Nephritis, nephrotic syndrome and nephrosis (N00-N07,N17-N19,N25-N27)': 'Kidney disease',
}

# Assign colors
PALETTE = ['#003049', '#005F73', '#0A9396', '#94D2BD', '#E9D8A6',
           '#EE9B00', '#CA6702', '#BB3E03', '#6B7280', '#374151']
LINE_COLORS = {}
for i, cause_raw in enumerate(ever_top5):
    LINE_COLORS[cause_raw] = PALETTE[i % len(PALETTE)]

# Plot
fig, ax = plt.subplots(figsize=(14, 7))

for cause_raw in ever_top5:
    cause_name = CAUSE_NAMES.get(cause_raw, cause_raw[:30])
    subset = top5_dynamic[top5_dynamic['cause'] == cause_raw].sort_values('year')
    years = subset['year'].values
    rates = subset['crude_rate'].values

    # Break into contiguous segments
    segments_x = []
    segments_y = []
    seg_x = [years[0]]
    seg_y = [rates[0]]
    for j in range(1, len(years)):
        if years[j] == years[j-1] + 1:
            seg_x.append(years[j])
            seg_y.append(rates[j])
        else:
            segments_x.append(seg_x)
            segments_y.append(seg_y)
            seg_x = [years[j]]
            seg_y = [rates[j]]
    segments_x.append(seg_x)
    segments_y.append(seg_y)

    for k, (sx, sy) in enumerate(zip(segments_x, segments_y)):
        ax.plot(sx, sy, linewidth=2.5, color=LINE_COLORS[cause_raw],
                label=cause_name if k == 0 else None)

# Abortion rate line (dashed)
abort_sorted = abort_trend.sort_values('year')
ax.plot(abort_sorted['year'], abort_sorted['rate_per_100k'],
        linewidth=3, color='#AE2012', linestyle='--',
        label='Abortion (if counted)', zorder=10)

# Style
ax.set_xlabel('')
ax.set_ylabel('Rate per 100,000 population', fontsize=12, color='#374151')
ax.set_title('Top 5 Causes of Death Each Year + Abortion (Rate per 100k, 1999-2024)',
             fontsize=14, fontweight='bold', loc='left', color='#003049')
ax.set_xlim(1999, 2024)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.2)
ax.legend(loc='upper right', frameon=False, fontsize=10)

# Annotate key events
ax.axvline(x=2020, color='#6B7280', linestyle=':', alpha=0.5, linewidth=1)
ax.text(2020.2, ax.get_ylim()[1] * 0.95, 'COVID', fontsize=9, color='#6B7280', va='top')

ax.axvline(x=2022, color='#6B7280', linestyle=':', alpha=0.5, linewidth=1)
ax.text(2022.2, ax.get_ylim()[1] * 0.90, 'Dobbs', fontsize=9, color='#6B7280', va='top')

plt.tight_layout()
plt.savefig('outputs/trend_top5_rate_plus_abortion.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()

print('\nNotes:')
print('  - Rate per 100k controls for population growth (22% increase 1999-2024)')
print('  - Heart disease rate dropped 23% (260 -> 201 per 100k)')
print('  - Cancer rate dropped 7% (197 -> 182 per 100k)')
print('  - Accidents rate rose 65% (35 -> 58 per 100k) driven by overdoses')
print('  - Abortion rate: uses total population denominator for comparability')


## Cleanup


In [ ]:
con.close()
print('\n✓ Database connection closed')
print('\n✓✓✓ All visualizations complete')
print('\nGenerated:' )
print('  Chart 1: 01_national_without_vs_with.png')
print('  Chart 2: 02_causes_by_race.png')
print('  Chart 3: 03_causes_by_sex_national.png (stacked)')
print('  Chart 3b: 03b_national_without_vs_with_race_*.png (2 files: white, black)')
print('  Chart 4: 04_causes_sex_age_multiples_national.png')
print('  Chart 5: 05_causes_age_multiples_race_*.png (6 figures)')
print('  Chart 6: 06_causes_by_sex_race_*_stacked.png (2 files: white, black)')
